# Pipeline & Task Health Audit Notebook

Portable audit of Snowflake-native orchestration health: Tasks, Streams, and pipe/ingest
reliability. Maps to the JD's operational-support responsibilities — troubleshooting
incidents, resolving production issues, and supporting platform reliability — as the
day-to-day counterpart to the one-time structural audits. Run top to bottom. It surfaces:

1. **Task inventory & schedule overview** — every task, its schedule, and whether it's actually enabled
2. **Task failures** — recent failures, ranked by frequency and recency
3. **Task duration & drift** — tasks that are slow, getting slower, or routinely missing their schedule
4. **Task tree / dependency health** — root tasks vs. child tasks, and whether a broken parent silently stops a whole chain
5. **Stream staleness & lag** — streams whose offset is falling behind, risking data loss if the stream is ever dropped/recreated
6. **Pipe (Snowpipe) health** — ingest failures and latency
7. **A rolled-up recommendation summary**

### Prerequisites
- A role with `IMPORTED PRIVILEGES` on `SNOWFLAKE` (or `ACCOUNTADMIN`), plus visibility into
  the specific tasks/streams/pipes being audited (ownership or `MONITOR` privilege).
- Nothing here is destructive. Every cell reads.


In [ ]:
-- ============================================================
-- PARAMETERS
-- ============================================================
SET lookback_days = 14;
SET slow_task_seconds = 300;      -- flag tasks whose runs regularly exceed this duration
SET stream_stale_days = 7;        -- flag streams with no consuming activity in this many days

SELECT $lookback_days AS lookback_days, $slow_task_seconds AS slow_task_seconds, $stream_stale_days AS stream_stale_days;


## 1. Task inventory & schedule overview

Starting point: every task that exists, whether it's actually running (`STARTED` state, not
just defined), and its schedule or predecessor. A task that's `SUSPENDED` but shouldn't be —
or vice versa — is one of the most common "silent failure" patterns, since a suspended task
simply stops producing errors instead of raising one.


In [ ]:
-- All tasks and their current state
SHOW TASKS IN ACCOUNT;


In [ ]:
-- Parse the SHOW TASKS output for a quick state overview
SELECT
    "database_name",
    "schema_name",
    "name"        AS task_name,
    "state",
    "schedule",
    "predecessors",
    "warehouse",
    "owner",
    "created_on"
FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()))
ORDER BY "state", "database_name", "schema_name", task_name;


In [ ]:
-- Tasks that exist in ACCOUNT_USAGE but haven't run at all in the lookback window
-- (distinct from "suspended" — this catches tasks that are enabled but not firing, e.g.
-- a broken cron schedule or a predecessor that never completes)
WITH recent_runs AS (
    SELECT DISTINCT database_name, schema_name, name
    FROM snowflake.account_usage.task_history
    WHERE query_start_time >= DATEADD(day, -$lookback_days, CURRENT_TIMESTAMP())
)
SELECT
    t.database_name,
    t.schema_name,
    t.name AS task_name,
    t.state,
    t.schedule,
    t.created_on,
    t.last_committed_on
FROM snowflake.account_usage.tasks t
WHERE t.deleted IS NULL
  AND t.state = 'started'
  AND NOT EXISTS (
        SELECT 1 FROM recent_runs r
        WHERE r.database_name = t.database_name AND r.schema_name = t.schema_name AND r.name = t.name
      )
ORDER BY t.database_name, t.schema_name, task_name;


## 2. Task failures

Ranked by failure count and recency so the noisiest and most recently broken tasks surface
first. `ERROR_MESSAGE` is truncated for readability — pull the full message for anything
you're actively triaging.


In [ ]:
-- Failed task runs in the lookback window
SELECT
    database_name,
    schema_name,
    name AS task_name,
    query_id,
    state,
    error_code,
    LEFT(error_message, 300) AS error_message_preview,
    scheduled_time,
    query_start_time,
    completed_time,
    DATEDIFF('second', query_start_time, completed_time) AS duration_sec
FROM snowflake.account_usage.task_history
WHERE query_start_time >= DATEADD(day, -$lookback_days, CURRENT_TIMESTAMP())
  AND state = 'FAILED'
ORDER BY query_start_time DESC
LIMIT 300;


In [ ]:
-- Failure rate by task — the tasks most worth fixing first
SELECT
    database_name,
    schema_name,
    name AS task_name,
    COUNT(*)                                            AS total_runs,
    SUM(IFF(state = 'FAILED', 1, 0))                    AS failed_runs,
    SUM(IFF(state = 'SUCCEEDED', 1, 0))                 AS succeeded_runs,
    ROUND(100.0 * SUM(IFF(state = 'FAILED', 1, 0)) / COUNT(*), 1) AS failure_rate_pct,
    MAX(IFF(state = 'FAILED', query_start_time, NULL))  AS last_failure
FROM snowflake.account_usage.task_history
WHERE query_start_time >= DATEADD(day, -$lookback_days, CURRENT_TIMESTAMP())
GROUP BY database_name, schema_name, task_name
HAVING SUM(IFF(state = 'FAILED', 1, 0)) > 0
ORDER BY failure_rate_pct DESC, failed_runs DESC;


## 3. Task duration & drift

A task that's gradually taking longer each run is an early warning — it'll eventually miss
its schedule window or start overlapping with the next scheduled run, before it ever shows up
as an outright failure.


In [ ]:
-- Task duration trend by day, for tasks with meaningful runtime
SELECT
    database_name,
    schema_name,
    name AS task_name,
    DATE_TRUNC('day', query_start_time) AS run_date,
    COUNT(*)                                                   AS run_count,
    ROUND(AVG(DATEDIFF('second', query_start_time, completed_time)), 1) AS avg_duration_sec,
    ROUND(MAX(DATEDIFF('second', query_start_time, completed_time)), 1) AS max_duration_sec
FROM snowflake.account_usage.task_history
WHERE query_start_time >= DATEADD(day, -$lookback_days, CURRENT_TIMESTAMP())
  AND state = 'SUCCEEDED'
GROUP BY database_name, schema_name, task_name, run_date
HAVING AVG(DATEDIFF('second', query_start_time, completed_time)) > $slow_task_seconds
ORDER BY task_name, run_date;


In [ ]:
-- Tasks whose runs are overlapping their own next scheduled start (schedule can't keep up)
WITH runs AS (
    SELECT
        database_name, schema_name, name AS task_name,
        query_start_time, completed_time,
        LEAD(query_start_time) OVER (PARTITION BY database_name, schema_name, name ORDER BY query_start_time) AS next_start
    FROM snowflake.account_usage.task_history
    WHERE query_start_time >= DATEADD(day, -$lookback_days, CURRENT_TIMESTAMP())
      AND state = 'SUCCEEDED'
)
SELECT
    database_name, schema_name, task_name,
    query_start_time, completed_time, next_start,
    DATEDIFF('second', completed_time, next_start) AS gap_before_next_run_sec
FROM runs
WHERE next_start IS NOT NULL
  AND completed_time > next_start   -- this run finished AFTER the next one already started
ORDER BY query_start_time DESC
LIMIT 100;


## 4. Task tree / dependency health

A single broken root task can silently stop an entire downstream chain — every child task
just never fires, without individually erroring. Mapping the tree makes that kind of failure
visible instead of requiring someone to already know the dependency structure by memory.


In [ ]:
-- Root tasks (no predecessor) and how many children depend on them
WITH tree AS (
    SELECT "database_name", "schema_name", "name" AS task_name, "predecessors", "state"
    FROM TABLE(RESULT_SCAN((SELECT LAST_QUERY_ID(-2))))  -- reuses the SHOW TASKS result from Section 1 if re-run adjacently; otherwise re-run: SHOW TASKS IN ACCOUNT;
)
SELECT task_name, database_name, schema_name, state, predecessors
FROM tree
WHERE predecessors = '[]' OR predecessors IS NULL
ORDER BY database_name, schema_name, task_name;


*(If the cell above errors because `SHOW TASKS` wasn't run immediately before it in this
session, just re-run `SHOW TASKS IN ACCOUNT;` directly above it — `RESULT_SCAN` needs the
`SHOW` command's result still in scope.)*


In [ ]:
-- Children whose predecessor task is currently suspended or has been failing —
-- the clearest "silently broken chain" signal
WITH failing_or_suspended AS (
    SELECT DISTINCT database_name, schema_name, name
    FROM snowflake.account_usage.tasks
    WHERE deleted IS NULL AND state = 'suspended'
)
SELECT
    t.database_name,
    t.schema_name,
    t.name AS task_name,
    t.predecessors,
    t.state AS child_state
FROM snowflake.account_usage.tasks t
WHERE t.deleted IS NULL
  AND t.predecessors IS NOT NULL
  AND t.predecessors != '[]'
  AND EXISTS (
        SELECT 1 FROM failing_or_suspended f
        WHERE t.predecessors ILIKE '%' || f.name || '%'
      )
ORDER BY t.database_name, t.schema_name, task_name;


## 5. Stream staleness & lag

Streams track changes since they were last consumed. If nothing has consumed a stream in a
long time, that's usually either an abandoned pipeline (safe to clean up) or a broken one
(the consuming task/process silently stopped) — and if the stream's underlying table data
falls outside Time Travel retention before it's consumed, the stream becomes stale and needs
to be recreated, losing the unconsumed change data entirely.


In [ ]:
-- All streams and their staleness status
SHOW STREAMS IN ACCOUNT;


In [ ]:
-- Parse for streams flagged stale, or with no recent consuming activity
SELECT
    "database_name",
    "schema_name",
    "name"        AS stream_name,
    "table_name",
    "type",
    "stale"       AS is_stale,
    "stale_after",
    "owner",
    "created_on"
FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()))
ORDER BY ("stale" = 'true') DESC, "stale_after" ASC NULLS LAST;


`is_stale = true` means the change data is already lost and the stream must be recreated
before it's useful again — treat every row where that's true as an active incident, not a
backlog item. `stale_after` on non-stale streams tells you the deadline to consume before that
happens; sort by soonest deadline to prioritize.


## 6. Pipe (Snowpipe) health

Ingest failures are easy to miss because, unlike a task, a failed pipe load doesn't
necessarily break anything downstream immediately — data just quietly doesn't arrive.


In [ ]:
-- Recent pipe load history: failures and latency
SELECT
    pipe_catalog_name AS database_name,
    pipe_schema_name  AS schema_name,
    pipe_name,
    file_name,
    status,
    LEFT(first_error_message, 300) AS first_error_message_preview,
    row_count,
    row_parsed,
    error_count,
    last_load_time
FROM snowflake.account_usage.copy_history
WHERE last_load_time >= DATEADD(day, -$lookback_days, CURRENT_TIMESTAMP())
  AND pipe_name IS NOT NULL
  AND (status != 'LOADED' OR error_count > 0)
ORDER BY last_load_time DESC
LIMIT 300;


In [ ]:
-- Pipe volume and error rate, rolled up
SELECT
    pipe_catalog_name AS database_name,
    pipe_schema_name  AS schema_name,
    pipe_name,
    COUNT(*)                                          AS files_processed,
    SUM(IFF(status = 'LOADED', 1, 0))                 AS files_loaded,
    SUM(IFF(status != 'LOADED', 1, 0))                AS files_failed,
    ROUND(100.0 * SUM(IFF(status != 'LOADED', 1, 0)) / COUNT(*), 1) AS failure_rate_pct,
    MAX(last_load_time)                                AS last_activity
FROM snowflake.account_usage.copy_history
WHERE last_load_time >= DATEADD(day, -$lookback_days, CURRENT_TIMESTAMP())
  AND pipe_name IS NOT NULL
GROUP BY database_name, schema_name, pipe_name
ORDER BY failure_rate_pct DESC, files_failed DESC;


In [ ]:
-- Pipes defined in the account that show no load activity at all in the window
-- (could be idle-by-design, or could be a silently broken ingest path)
SELECT
    "database_name",
    "schema_name",
    "name" AS pipe_name,
    "definition",
    "owner",
    "notification_channel"
FROM TABLE(RESULT_SCAN((SELECT LAST_QUERY_ID(-3))))


*(Same note as Section 4: if the cell above errors, run `SHOW PIPES IN ACCOUNT;` directly
above it first, then cross-reference its output against the `pipe_name` values in
`pipe_failure_rate` above to find pipes with zero recent activity.)*


## 7. Rolled-up recommendation summary


In [ ]:
-- Per-task summary combining failure rate, duration, and current state
SELECT
    t.database_name,
    t.schema_name,
    t.name AS task_name,
    t.state,
    COALESCE(f.total_runs, 0)         AS total_runs_in_window,
    COALESCE(f.failed_runs, 0)        AS failed_runs,
    COALESCE(f.failure_rate_pct, 0)   AS failure_rate_pct,
    f.last_failure,
    ARRAY_TO_STRING(ARRAY_CONSTRUCT_COMPACT(
        IFF(t.state = 'started' AND COALESCE(f.total_runs, 0) = 0, 'ENABLED BUT NOT RUNNING — investigate schedule/predecessor', NULL),
        IFF(COALESCE(f.failure_rate_pct, 0) > 25, 'HIGH FAILURE RATE — prioritize triage', NULL),
        IFF(COALESCE(f.failure_rate_pct, 0) BETWEEN 1 AND 25, 'INTERMITTENT FAILURES — review error pattern', NULL)
    ), '; ') AS recommendations
FROM snowflake.account_usage.tasks t
LEFT JOIN (
    SELECT database_name, schema_name, name,
           COUNT(*) AS total_runs,
           SUM(IFF(state = 'FAILED', 1, 0)) AS failed_runs,
           ROUND(100.0 * SUM(IFF(state = 'FAILED', 1, 0)) / COUNT(*), 1) AS failure_rate_pct,
           MAX(IFF(state = 'FAILED', query_start_time, NULL)) AS last_failure
    FROM snowflake.account_usage.task_history
    WHERE query_start_time >= DATEADD(day, -$lookback_days, CURRENT_TIMESTAMP())
    GROUP BY database_name, schema_name, name
) f ON t.database_name = f.database_name AND t.schema_name = f.schema_name AND t.name = f.name
WHERE t.deleted IS NULL
ORDER BY (t.state = 'started' AND COALESCE(f.total_runs,0) = 0) DESC, failure_rate_pct DESC;


### Notes, caveats, and next steps

- **`SHOW TASKS`/`SHOW STREAMS`/`SHOW PIPES` only return objects the current role can see.**
  Run this with a role that has broad visibility (or `MONITOR` on the relevant
  databases/schemas), or the inventory will be incomplete without any error telling you so.
- **`RESULT_SCAN(LAST_QUERY_ID(-N))`** is used in a couple of cells to reuse an earlier `SHOW`
  result — this is order-dependent within a session. If cells get re-run out of order, just
  re-run the `SHOW` command immediately before the affected cell.
- **A high task failure rate and a suspended predecessor are both "loud."** The quieter, more
  dangerous findings are usually Section 1's "enabled but not running" list and Section 5's
  stale streams — both fail silently by design, which is exactly why they're worth checking
  deliberately rather than waiting for someone to notice missing data downstream.
- **Pair with the warehouse right-sizing notebook** for tasks — a task's `warehouse` assignment
  is exactly the kind of thing that notebook's Section 5 (small workloads on large warehouses)
  can catch if the task's queries show up there.
